# Integer Linear Programming (ILP) Solutions

This notebook contains the Python (`PuLP 4.0`) code and verified solutions for all Integer Linear Programming, Mixed-Integer Linear Programming (MILP), and Sensitivity Analysis examples presented in the `Integer_Linear_Programming.tex` slide deck for **BUS 104**.

### 1. Simple Pure Integer Linear Program & Branch-and-Bound

**Problem Formulation:**
$$\max z = 3x_1 + 2x_2$$
subject to:
$$\begin{aligned}
2x_1 + x_2 &\le 7 \quad (C_1) \\
-x_1 + x_2 &\le 3 \quad (C_2) \\
x_1, x_2 &\in \mathbb{Z}_{\ge 0}
\end{aligned}$$

The cell below runs the exact `PuLP 4.0` code from **Section 1** of the slides (`Simple_Pure_ILP`).

In [1]:
import pulp

# 1. Initialize Pure ILP Model (PuLP 4.0 API)
prob = pulp.LpProblem("Simple_Pure_ILP", pulp.LpMaximize)

# 2. Integer Decision Variables: x1, x2 in Z_{>= 0}
x1 = prob.add_variable("x1", lowBound=0, cat=pulp.LpInteger)
x2 = prob.add_variable("x2", lowBound=0, cat=pulp.LpInteger)

# 3. Objective Function
prob += 3 * x1 + 2 * x2, "Total_Profit"

# 4. Resource Constraints
prob += 2 * x1 + x2 <= 7, "C1"
prob += -x1 + x2 <= 3, "C2"

# 5. Solve and inspect Result
res = prob.solve()
print(f"Status: {res.status.name}")
print(f"Optimal z* = {pulp.value(prob.objective):.2f}")
print(f"x1* = {x1.varValue}, x2* = {x2.varValue}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/fecd4965585c4789a6dd8d6acf905fc7-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/fecd4965585c4789a6dd8d6acf905fc7-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 7 COLUMNS
At line 18 RHS
At line 21 BOUNDS
At line 24 ENDATA
Problem MODEL has 2 rows, 2 columns and 4 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -12.6667 - 0.00 seconds
Cgl0004I processed model has 2 rows, 2 columns (2 integer (0 of which binary)) and 4 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of -11 found by DiveCoefficient after 0 iterations and 0 nodes (0.00 seconds)
Cbc0012I Integer solution of -12 found by DiveCoefficient after 1 iterations and 0 nodes (0.00 sec

#### 1b. Comparing the Continuous LP Relaxation vs. Pure ILP (Why Naive Rounding Fails)

In the continuous LP relaxation ($x_1, x_2 \ge 0$), the optimal solution is $(x_1^*, x_2^*) = (4/3, 13/3) \approx (1.33, 4.33)$ with $z_{LP}^* = 12.67$. Rounding down to $(1, 4)$ yields $z = 11.00$ (suboptimal), while rounding to $(1, 5)$, $(2, 4)$, or $(2, 5)$ violates the constraints. Branch-and-Bound finds the true integer optimum $(2, 3)$ with $z^* = 12.00$.

In [2]:
import pulp

# Solve the Continuous LP Relaxation
lp_relax = pulp.LpProblem("LP_Relaxation", pulp.LpMaximize)
x1_c = lp_relax.add_variable("x1", lowBound=0, cat=pulp.LpContinuous)
x2_c = lp_relax.add_variable("x2", lowBound=0, cat=pulp.LpContinuous)

lp_relax += 3 * x1_c + 2 * x2_c, "Total_Profit"
lp_relax += 2 * x1_c + x2_c <= 7, "C1"
lp_relax += -x1_c + x2_c <= 3, "C2"

lp_relax.solve(pulp.COIN_CMD(msg=False))
print(f"LP Relaxation Optimum : x1* = {x1_c.varValue:.4f}, x2* = {x2_c.varValue:.4f}, z_LP* = {pulp.value(lp_relax.objective):.4f}")
print(f"Naive Rounded (1, 4)  : x1  = 1.0000, x2  = 4.0000, z     = {3*1 + 2*4:.4f} (Suboptimal)")
print(f"True Pure ILP Optimum : x1* = 2.0000, x2* = 3.0000, z_ILP*= {3*2 + 2*3:.4f} (Optimal)")

LP Relaxation Optimum : x1* = 1.3333, x2* = 4.3333, z_LP* = 12.6667
Naive Rounded (1, 4)  : x1  = 1.0000, x2  = 4.0000, z     = 11.0000 (Suboptimal)
True Pure ILP Optimum : x1* = 2.0000, x2* = 3.0000, z_ILP*= 12.0000 (Optimal)


### 2. The 0-1 Knapsack Problem

**Problem Context:**
A hiker can carry at most $W = 10\text{ kg}$ and must choose which items to pack ($x_i \in \{0, 1\}$):
- **Laptop:** Value $= \$40$, Weight $= 4\text{ kg}$ (Ratio $= \$10.00/\text{kg}$)
- **Gold:** Value $= \$50$, Weight $= 3\text{ kg}$ (Ratio $= \$16.67/\text{kg}$)
- **Silver:** Value $= \$30$, Weight $= 6\text{ kg}$ (Ratio $= \$5.00/\text{kg}$)
- **Books:** Value $= \$10$, Weight $= 5\text{ kg}$ (Ratio $= \$2.00/\text{kg}$)

The cell below runs the exact `PuLP 4.0` code from **Section 2** of the slides (`Knapsack_0_1`).

In [3]:
import pulp

items = ["Laptop", "Gold", "Silver", "Books"]
val = {"Laptop": 40, "Gold": 50, "Silver": 30, "Books": 10}
wt  = {"Laptop": 4,  "Gold": 3,  "Silver": 6,  "Books": 5}

prob = pulp.LpProblem("Knapsack_0_1", pulp.LpMaximize)

# Binary variables x[i] in {0, 1} using PuLP 4.0 API
x = prob.add_variable_dicts("x", items, cat=pulp.LpBinary)

# Objective & Weight Capacity Constraint
prob += pulp.lpSum(val[i] * x[i] for i in items), "Total_Value"
prob += pulp.lpSum(wt[i] * x[i] for i in items) <= 10, "Max_Weight"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Optimal Value = ${z_opt:.0f}")
for i in items:
    if x[i].varValue > 0.5:
        print(f"  Take {i}: weight={wt[i]} kg, value=${val[i]}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/230bb4812e13441e8e8eed75a137f55f-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/230bb4812e13441e8e8eed75a137f55f-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 6 COLUMNS
At line 23 RHS
At line 25 BOUNDS
At line 30 ENDATA
Problem MODEL has 1 rows, 4 columns and 4 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -105 - 0.00 seconds
Cgl0004I processed model has 1 rows, 4 columns (4 integer (4 of which binary)) and 4 elements
Cutoff increment increased from 1e-05 to 9.9999
Cbc0038I Initial state - 1 integers unsatisfied sum - 0.5
Cbc0038I Solution found of -90
Cbc0038I Before mini branch and bound, 3 integers at bound fixed and 0 continuous
Cbc0038I Full problem 1 rows 4 colu

#### 2b. When Does the Greedy Heuristic Fail? (Counterexample from Slides)

Consider a knapsack with capacity $W = 6\text{ kg}$ and three items:
- **Item 1:** Value $= \$32$, Weight $= 4\text{ kg}$ (Ratio $= \$8.00/\text{kg}$)
- **Item 2:** Value $= \$22$, Weight $= 3\text{ kg}$ (Ratio $= \$7.33/\text{kg}$)
- **Item 3:** Value $= \$22$, Weight $= 3\text{ kg}$ (Ratio $= \$7.33/\text{kg}$)

The greedy heuristic selects **Item 1** first (highest ratio $\$8.00/\text{kg}$), leaving $2\text{ kg}$ of unused capacity and earning **$z_{\text{greedy}} = \$32$**. The 0-1 ILP solver selects **Items 2 and 3**, filling all $6\text{ kg}$ and earning **$z^* = \$44$**.

In [4]:
import pulp

items_ce = ["Item_1", "Item_2", "Item_3"]
val_ce = {"Item_1": 32, "Item_2": 22, "Item_3": 22}
wt_ce  = {"Item_1": 4,  "Item_2": 3,  "Item_3": 3}

prob_ce = pulp.LpProblem("Knapsack_Greedy_Counterexample", pulp.LpMaximize)
x_ce = prob_ce.add_variable_dicts("x", items_ce, cat=pulp.LpBinary)

prob_ce += pulp.lpSum(val_ce[i] * x_ce[i] for i in items_ce), "Total_Value"
prob_ce += pulp.lpSum(wt_ce[i] * x_ce[i] for i in items_ce) <= 6, "Capacity_6kg"

prob_ce.solve(pulp.COIN_CMD(msg=False))
print(f"Greedy Heuristic Choice : ['Item_1']          -> Weight = 4 kg, Value = $32")
print(f"Optimal 0-1 ILP Choice  : {[i for i in items_ce if x_ce[i].varValue > 0.5]} -> Weight = 6 kg, Value = ${pulp.value(prob_ce.objective):.0f}")

Greedy Heuristic Choice : ['Item_1']          -> Weight = 4 kg, Value = $32
Optimal 0-1 ILP Choice  : ['Item_2', 'Item_3'] -> Weight = 6 kg, Value = $44


### 3. 0-1 Knapsack with Logical Constraints

**Logical Conditions Added to the 4-Item Knapsack Problem:**
1. **Mutual Exclusion:** Cannot take both Laptop and Gold ($x_{\text{Laptop}} + x_{\text{Gold}} \le 1$).
2. **Implication (If-Then):** If Gold is taken, Books must also be taken ($x_{\text{Gold}} - x_{\text{Books}} \le 0$).

The cell below runs the exact `PuLP 4.0` code from **Section 3** of the slides (`Knapsack_Logical`).

In [5]:
import pulp

items = ["Laptop", "Gold", "Silver", "Books"]
val = {"Laptop": 40, "Gold": 50, "Silver": 30, "Books": 10}
wt  = {"Laptop": 4,  "Gold": 3,  "Silver": 6,  "Books": 5}

prob = pulp.LpProblem("Knapsack_Logical", pulp.LpMaximize)
x = prob.add_variable_dicts("x", items, cat=pulp.LpBinary)

prob += pulp.lpSum(val[i] * x[i] for i in items), "Total_Value"
prob += pulp.lpSum(wt[i] * x[i] for i in items) <= 10, "Max_Weight"

# Rule 1 (Mutual Exclusion): Cannot take both Laptop and Gold
prob += x["Laptop"] + x["Gold"] <= 1, "Mutually_Exclusive"

# Rule 2 (Implication): If Gold is taken, Books must be taken
prob += x["Gold"] - x["Books"] <= 0, "Gold_Implies_Books"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Optimal Value = ${z_opt:.0f}")
print("Selected:", [i for i in items if x[i].varValue > 0.5])

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/6e6eae66242f488d87f1d23284a4dcbb-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/6e6eae66242f488d87f1d23284a4dcbb-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 8 COLUMNS
At line 29 RHS
At line 33 BOUNDS
At line 38 ENDATA
Problem MODEL has 3 rows, 4 columns and 8 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -70 - 0.00 seconds
Cgl0004I processed model has 3 rows, 4 columns (4 integer (4 of which binary)) and 8 elements
Cutoff increment increased from 1e-05 to 9.9999
Cbc0038I Initial state - 0 integers unsatisfied sum - 1.11022e-16
Cbc0038I Solution found of -70
Cbc0038I Before mini branch and bound, 4 integers at bound fixed and 0 continuous
Cbc0038I Mini branch and bou

### 4. Fixed Setup Cost Problem (Mixed-Integer Linear Program)

**Problem Formulation:**
$$\max Z = 5x_1 + 3x_2 - 10y$$
subject to:
$$\begin{aligned}
x_1 + x_2 &\le 10 \quad (\text{Resource A}) \\
2x_1 + x_2 &\le 16 \quad (\text{Resource B}) \\
x_2 &\le 7 \quad (\text{Resource C}) \\
x_1 + x_2 - 10y &\le 0 \quad (\text{Big-}M \text{ Linking Constraint, } M = 10) \\
x_1, x_2 &\ge 0, \quad y \in \{0, 1\}
\end{aligned}$$

The cell below runs the exact `PuLP 4.0` code from **Section 4** of the slides (`Fixed_Setup_Cost_MILP`).

In [6]:
import pulp

prob = pulp.LpProblem("Fixed_Setup_Cost_MILP", pulp.LpMaximize)

# Continuous production variables and binary setup variable
x1 = prob.add_variable("x1", lowBound=0, cat=pulp.LpContinuous)
x2 = prob.add_variable("x2", lowBound=0, cat=pulp.LpContinuous)
y  = prob.add_variable("y", cat=pulp.LpBinary)

# Objective: Gross Profit - Fixed Setup Cost
prob += 5 * x1 + 3 * x2 - 10 * y, "Net_Profit"

# Resource constraints + Big-M linking constraint (M = 10)
prob += x1 + x2 <= 10, "Resource_A"
prob += 2 * x1 + x2 <= 16, "Resource_B"
prob += x2 <= 7, "Resource_C"
prob += x1 + x2 - 10 * y <= 0, "Big_M_Link"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Net Profit Z* = ${z_opt:.2f}")
print(f"Activate y*={int(y.varValue)}, x1*={x1.varValue}, x2*={x2.varValue}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/e33a4f7fcdde4e5ca4b463c763493d16-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/e33a4f7fcdde4e5ca4b463c763493d16-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 9 COLUMNS
At line 23 RHS
At line 28 BOUNDS
At line 30 ENDATA
Problem MODEL has 4 rows, 3 columns and 8 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -32 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 2 strengthened rows, 0 substitutions
Cgl0004I processed model has 2 rows, 3 columns (1 integer (1 of which binary)) and 6 elements
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of -32
Cbc0038I Relaxing continuous gives -32
Cbc0038I Before mini branch and bound, 1 integers at b

### 5. Minimum Purchase Quantity (MOQ) Problem

**Problem Formulation:**
A firm may place an order with a supplier ($y \in \{0, 1\}$) with a fixed setup fee of $\$12$, a single resource constraint ($2x_1 + x_2 \le 16$ assembly hours), and a **two-sided switching rule** ($6y \le x_1 + x_2 \le 16y$) enforcing a Minimum Order Quantity (MOQ) of $6$ total units whenever $y = 1$:
$$\max Z = 7x_1 + 5x_2 - 12y$$
subject to:
$$\begin{aligned}
2x_1 + x_2 &\le 16 \quad (\text{Assembly Hours}) \\
x_1 + x_2 - 6y &\ge 0 \quad (\text{MOQ Lower Switch: } x_1 + x_2 \ge 6 \text{ if } y = 1) \\
x_1 + x_2 - 16y &\le 0 \quad (\text{Big-}M \text{ Upper Switch: } x_1 + x_2 = 0 \text{ if } y = 0) \\
x_1, x_2 &\in \mathbb{Z}_{\ge 0}, \quad y \in \{0, 1\}
\end{aligned}$$

The cell below runs the exact `PuLP 4.0` code from **Section 5** of the slides (`Supplier_Order_MOQ`).

In [7]:
import pulp

prob = pulp.LpProblem("Supplier_Order_MOQ", pulp.LpMaximize)

# Integer order quantities x1, x2 and binary order indicator y
x1 = prob.add_variable("x1", lowBound=0, cat=pulp.LpInteger)
x2 = prob.add_variable("x2", lowBound=0, cat=pulp.LpInteger)
y  = prob.add_variable("y", cat=pulp.LpBinary)

prob += 7 * x1 + 5 * x2 - 12 * y, "Net_Profit"

# Single resource constraint
prob += 2 * x1 + x2 <= 16, "Assembly_Hours"

# Two-sided switching constraints: 6*y <= x1 + x2 <= 16*y
prob += x1 + x2 - 6 * y >= 0, "MOQ_Lower_Switch"
prob += x1 + x2 - 16 * y <= 0, "Big_M_Upper_Switch"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Profit Z* = ${z_opt:.2f}")
print(f"y*={y.varValue:.0f}, x1*={x1.varValue:.0f}, x2*={x2.varValue:.0f}")

Status: Optimal, Profit Z* = $68.00
y*=1, x1*=0, x2*=16


### 6. Fixed-Charge Transshipment Problem

**Problem Formulation:**
Three warehouses ($W_1, W_2, W_3$), each holding $S_i = 50$ units, can ship through a central Transshipment Hub ($T$) to satisfy Customer Sink ($D$) demand of $80$ units. Activating Warehouse $W_i$ incurs a fixed setup fee $F_i$ ($\$120, \$70, \$50$) plus a per-unit inbound cost $c_{iT}$ ($\$2, \$4, \$5$), and Hub $T$ forwards to Customer $D$ at $\$3/\text{unit}$:
$$\min Z = 120y_1 + 70y_2 + 50y_3 + 2x_{1T} + 4x_{2T} + 5x_{3T} + 3x_{TD}$$
subject to:
$$\begin{aligned}
x_{iT} - 50y_i &\le 0 \quad (i = 1, 2, 3: \text{Big-}M \text{ Warehouse Capacity \& Activation}) \\
x_{1T} + x_{2T} + x_{3T} - x_{TD} &= 0 \quad (\text{Flow Conservation at Hub } T) \\
x_{TD} &= 80 \quad (\text{Demand at Sink } D) \\
x_{iT}, x_{TD} &\ge 0, \quad y_1, y_2, y_3 \in \{0, 1\}
\end{aligned}$$

The cell below runs the exact `PuLP 4.0` code from **Section 6** of the slides (`Fixed_Charge_Transshipment`).

In [8]:
import pulp

prob = pulp.LpProblem("Fixed_Charge_Transshipment", pulp.LpMinimize)
x1T = prob.add_variable("x1T", lowBound=0, cat=pulp.LpContinuous)
x2T = prob.add_variable("x2T", lowBound=0, cat=pulp.LpContinuous)
x3T = prob.add_variable("x3T", lowBound=0, cat=pulp.LpContinuous)
xTD = prob.add_variable("xTD", lowBound=0, cat=pulp.LpContinuous)
y1  = prob.add_variable("y1", cat=pulp.LpBinary)
y2  = prob.add_variable("y2", cat=pulp.LpBinary)
y3  = prob.add_variable("y3", cat=pulp.LpBinary)

prob += 120*y1 + 70*y2 + 50*y3 + 2*x1T + 4*x2T + 5*x3T + 3*xTD
prob += x1T - 50 * y1 <= 0, "Cap_W1"
prob += x2T - 50 * y2 <= 0, "Cap_W2"
prob += x3T - 50 * y3 <= 0, "Cap_W3"
prob += x1T + x2T + x3T - xTD == 0, "Hub_T_Balance"
prob += xTD == 80, "Sink_D_Demand"

res = prob.solve()
print(f"Status: {res.status.name}, Cost Z* = ${pulp.value(prob.objective):.2f}")
print(f"Open: y1={y1.varValue:.0f}, y2={y2.varValue:.0f}, y3={y3.varValue:.0f}")
print(f"Flow: x1T={x1T.varValue:.0f}, x2T={x2T.varValue:.0f}, xTD={xTD.varValue:.0f}")

Status: Optimal, Cost Z* = $650.00
Open: y1=1, y2=1, y3=0
Flow: x1T=50, x2T=30, xTD=80


### 7. Store Location Problem: Set Covering on a $2 \times 3$ City Grid

**0-1 Set Covering Model:**
$$\min Z = x_1 + x_2 + x_3 + x_4 + x_5 + x_6$$
subject to:
$$\begin{aligned}
x_1 + x_2 + x_4 &\ge 1 \quad (\text{Cover Sector 1}) \\
x_1 + x_2 + x_3 + x_5 &\ge 1 \quad (\text{Cover Sector 2}) \\
x_2 + x_3 + x_6 &\ge 1 \quad (\text{Cover Sector 3}) \\
x_1 + x_4 + x_5 &\ge 1 \quad (\text{Cover Sector 4}) \\
x_2 + x_4 + x_5 + x_6 &\ge 1 \quad (\text{Cover Sector 5}) \\
x_3 + x_5 + x_6 &\ge 1 \quad (\text{Cover Sector 6}) \\
x_1, \dots, x_6 &\in \{0, 1\}
\end{aligned}$$

- **Optimal 0-1 Solution:** $Z^* = 2$ stores (with alternative optima at $\{2, 5\}$ Central Spine, $\{1, 6\}$ Opposite Corners, and $\{3, 4\}$ Opposite Corners).

The cell below runs the exact `PuLP 4.0` code from **Section 7** of the slides (`City_Grid_Set_Covering`).

In [9]:
import pulp

prob = pulp.LpProblem("City_Grid_Set_Covering", pulp.LpMinimize)

sectors = [1, 2, 3, 4, 5, 6]
neighbors = {
    1: [1, 2, 4],    2: [1, 2, 3, 5], 3: [2, 3, 6],
    4: [1, 4, 5],    5: [2, 4, 5, 6], 6: [3, 5, 6],
}

# Binary store location variables x[1] .. x[6]
x = {j: prob.add_variable(f"x_{j}", cat=pulp.LpBinary) for j in sectors}

# Objective: Minimize total number of stores opened
prob += sum(x[j] for j in sectors), "Total_Stores"

# Coverage constraint for each sector i
for i in sectors:
    prob += sum(x[j] for j in neighbors[i]) >= 1, f"Cover_Sector_{i}"

res = prob.solve()
chosen = [j for j in sectors if x[j].varValue > 0.5]
print(f"Status: {res.status.name}, Min Stores Z* = {pulp.value(prob.objective):.0f}")
print(f"Open Stores in Sectors: {chosen}")

Status: Optimal, Min Stores Z* = 2
Open Stores in Sectors: [3, 4]


### Q&A

- **Why can't we simply round the continuous LP relaxation solution to solve an Integer Linear Program (ILP)?**
  In Example 1 (`Simple_Pure_ILP`), the continuous LP relaxation optimum is $(x_1^*, x_2^*) = (1.33, 4.33)$ with $z_{LP}^* = 12.67$. Rounding to the nearest integer $(1, 4)$ gives $z = 11.00$, which is suboptimal, while rounding up violates the constraints. Branch-and-Bound identifies the true integer optimum at $(x_1^*, x_2^*) = (2, 3)$ with $z^* = 12.00$.
- **How do logical constraints and fixed setup costs change discrete optimization outcomes?**
  In the 4-item 0-1 Knapsack problem (Example 2), the unconstrained optimal selection is Laptop and Gold ($z^* = \$90$). Adding a mutual exclusion constraint between Laptop and Gold plus an implication constraint requiring Books whenever Gold is chosen (Example 3) shifts the optimal selection to Laptop and Silver ($z^* = \$70$).

### Data Analysis Key Findings

- **Pure ILP vs. LP Relaxation (Example 1):** Enforcing integrality reduces the maximum objective value from $z_{LP}^* = 12.67$ at $(1.33, 4.33)$ to $z_{ILP}^* = 12.00$ at $(2, 3)$.
- **0-1 Knapsack & Logical Constraints (Examples 2–3):** Under a $10\text{ kg}$ limit, packing Laptop ($4\text{ kg}, \$40$) and Gold ($3\text{ kg}, \$50$) achieves the maximum value of $\$90$. In the $6\text{ kg}$ counterexample (Example 2b), the greedy value-to-weight heuristic earns $\$32$, whereas the exact 0-1 ILP earns $\$44$.
- **Fixed Setup Cost & Minimum Order Quantity MILPs (Examples 4–5):** Using a tight Big-$M$ linking constraint ($x_1 + x_2 - 10y \le 0$) in Example 4 yields an optimal net profit of $Z^* = \$32.00$ at $(x_1^*, x_2^*, y^*) = (6.0, 4.0, 1)$. In the simplified MOQ problem (Example 5), enforcing the two-sided switching constraints $6y \le x_1 + x_2 \le 16y$ with $2x_1 + x_2 \le 16$ yields $Z^* = \$68.00$ at $(x_1^*, x_2^*, y^*) = (0, 16, 1)$.
- **Fixed-Charge Transshipment MILP (Example 6):** Combining binary warehouse setup fees ($F_1=\$120, F_2=\$70, F_3=\$50$) with Big-$M$ capacity activation ($x_{iT} - 50y_i \le 0$) and hub flow conservation ($x_{1T}+x_{2T}+x_{3T}-x_{TD}=0$) yields an optimal total cost of $Z^* = \$650.00$ by opening Warehouses 1 and 2 ($y_1^*=1, y_2^*=1, y_3^*=0$ and $x_{1T}^*=50, x_{2T}^*=30, x_{TD}^*=80$).
- **Grid Store Location / Set Covering (Example 7):** Covering all 6 sectors of a $2 \times 3$ city grid under orthogonal adjacency requires a minimum of $Z^* = 2$ stores (achieved via the Central Spine $\{2, 5\}$ or Opposite Corners $\{1, 6\}$ / $\{3, 4\}$).

### Insights or Next Steps

- **Always Verify Big-$M$ Bounds:** Choose the smallest valid upper bound $M$ implied by physical resource capacities ($M = 10$ in Example 4, $M = 16$ in Example 5, and $M = 50$ in Example 6) to keep the LP relaxation tight and accelerate Branch-and-Bound convergence.